In [1]:
# ---
# title: "Fase 1 — Adquisición y preprocesamiento en GEE"
# author: "Lina María Quintero Fonseca"
# format: html
# ---

# Importaciones
import ee
import geemap
import json

ee.Initialize()
print("GEE inicializado")

GEE inicializado


In [2]:
# Fase 1 — Adquisición y preprocesamiento en GEE
# Lina María Quintero Fonseca | Maestría en Geomática, UNAL

import ee
import geemap
import json

ee.Initialize()

# Polígono AOI detallado de la CGSM
aoi = ee.Geometry.Polygon([[
    [-74.8700, 11.0600], [-74.8200, 11.0700], [-74.7500, 11.0650],
    [-74.6800, 11.0600], [-74.6000, 11.0500], [-74.5300, 11.0400],
    [-74.4800, 11.0350], [-74.4300, 11.0300], [-74.3800, 11.0200],
    [-74.3200, 11.0100], [-74.2500, 10.9900], [-74.2000, 10.9500],
    [-74.1700, 10.9000], [-74.1500, 10.8500], [-74.1300, 10.8000],
    [-74.1200, 10.7500], [-74.1100, 10.7000], [-74.1000, 10.6500],
    [-74.1200, 10.6000], [-74.1500, 10.5500], [-74.2000, 10.5000],
    [-74.2500, 10.4500], [-74.3000, 10.4000], [-74.3500, 10.3500],
    [-74.4500, 10.3400], [-74.5500, 10.3800], [-74.6200, 10.4200],
    [-74.7000, 10.4800], [-74.7500, 10.5500], [-74.8000, 10.6500],
    [-74.8500, 10.7500], [-74.8700, 10.8500], [-74.8800, 10.9500],
    [-74.8700, 11.0600]
]])

# Verificar
area_km2 = aoi.area().divide(1e6).getInfo()
print(f"Área AOI: {area_km2:.1f} km²")

# Visualizar
Map = geemap.Map(center=[10.7, -74.5], zoom=10)
Map.addLayer(ee.Feature(aoi), {'color': '1B5E20'}, 'CGSM AOI')
Map

Área AOI: 5073.0 km²


Map(center=[10.7, -74.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [3]:
# Fase 1 — Adquisición y preprocesamiento en GEE
# Lina María Quintero Fonseca | Maestría en Geomática, UNAL

import ee
import geemap
import json

ee.Initialize()

# Polígono AOI detallado de la CGSM
aoi = ee.Geometry.Polygon([[
    [-74.8700, 11.0600], [-74.8200, 11.0700], [-74.7500, 11.0650],
    [-74.6800, 11.0600], [-74.6000, 11.0500], [-74.5300, 11.0400],
    [-74.4800, 11.0350], [-74.4300, 11.0300], [-74.3800, 11.0200],
    [-74.3200, 11.0100], [-74.2500, 10.9900], [-74.2000, 10.9500],
    [-74.1700, 10.9000], [-74.1500, 10.8500], [-74.1300, 10.8000],
    [-74.1200, 10.7500], [-74.1100, 10.7000], [-74.1000, 10.6500],
    [-74.1200, 10.6000], [-74.1500, 10.5500], [-74.2000, 10.5000],
    [-74.2500, 10.4500], [-74.3000, 10.4000], [-74.3500, 10.3500],
    [-74.4500, 10.3400], [-74.5500, 10.3800], [-74.6200, 10.4200],
    [-74.7000, 10.4800], [-74.7500, 10.5500], [-74.8000, 10.6500],
    [-74.8500, 10.7500], [-74.8700, 10.8500], [-74.8800, 10.9500],
    [-74.8700, 11.0600]
]])

# Verificar
area_km2 = aoi.area().divide(1e6).getInfo()
print(f"Área AOI: {area_km2:.1f} km²")

# Visualizar
Map = geemap.Map(center=[10.7, -74.5], zoom=10)
Map.addLayer(ee.Feature(aoi), {'color': '1B5E20'}, 'CGSM AOI')
Map

Área AOI: 5073.0 km²


Map(center=[10.7, -74.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [5]:
# Función de máscara de nubes Sentinel-2
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(
           qa.bitwiseAnd(cirrus_bit).eq(0))
    return image.updateMask(mask)

# Función de índices espectrales
def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    cmri = ndvi.subtract(ndwi).rename('CMRI')
    return image.addBands([ndvi, ndwi, cmri])

# Construir colección Sentinel-2 (2017-2025)
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filterDate('2017-01-01', '2025-12-31')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .map(mask_s2_clouds)
      .map(add_indices))

n_images = s2.size().getInfo()
print(f"Imágenes Sentinel-2 disponibles: {n_images}")

Imágenes Sentinel-2 disponibles: 792


In [6]:
# Composite RGB 2024 para visualización
rgb_2024 = (s2.filterDate('2024-01-01', '2024-12-31')
            .select(['B4', 'B3', 'B2'])
            .median()
            .clip(aoi))

# Composite NDVI 2024
ndvi_2024 = (s2.filterDate('2024-01-01', '2024-12-31')
             .select('NDVI')
             .median()
             .clip(aoi))

Map2 = geemap.Map(center=[10.7, -74.5], zoom=10)
Map2.addLayer(rgb_2024, {'min': 0, 'max': 3000, 'bands': ['B4', 'B3', 'B2']}, 'RGB 2024')
Map2.addLayer(ndvi_2024, {'min': -0.2, 'max': 0.8, 'palette': ['red', 'yellow', 'green', 'darkgreen']}, 'NDVI 2024')
Map2.addLayer(ee.Feature(aoi), {'color': 'white'}, 'AOI', opacity=0.3)
Map2

Map(center=[10.7, -74.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [7]:
# Generar composites trimestrales de NDVI, NDWI y CMRI
def quarterly_composite(year, quarter):
    month_start = (quarter - 1) * 3 + 1
    start = ee.Date.fromYMD(year, month_start, 1)
    end = start.advance(3, 'month')
    
    composite = (s2.filterDate(start, end)
                .select(['NDVI', 'NDWI', 'CMRI'])
                .median()
                .clip(aoi))
    
    n = s2.filterDate(start, end).size()
    
    return (composite
            .set('year', year)
            .set('quarter', quarter)
            .set('n_images', n)
            .set('system:time_start', start.millis()))

# Generar para 2017-2025
composites = []
for year in range(2017, 2026):
    for quarter in range(1, 5):
        composites.append(quarterly_composite(year, quarter))

quarterly_col = ee.ImageCollection(composites)
print(f"Composites trimestrales generados: {len(composites)}")

# Verificar cuántas imágenes tiene cada trimestre
for year in [2017, 2020, 2024]:
    for q in range(1, 5):
        ms = (q - 1) * 3 + 1
        start = ee.Date.fromYMD(year, ms, 1)
        end = start.advance(3, 'month')
        n = s2.filterDate(start, end).size().getInfo()
        print(f"  {year}-Q{q}: {n} imágenes")

Composites trimestrales generados: 36
  2017-Q1: 2 imágenes
  2017-Q2: 0 imágenes
  2017-Q3: 0 imágenes
  2017-Q4: 1 imágenes
  2020-Q1: 62 imágenes
  2020-Q2: 19 imágenes
  2020-Q3: 11 imágenes
  2020-Q4: 21 imágenes
  2024-Q1: 57 imágenes
  2024-Q2: 9 imágenes
  2024-Q3: 12 imágenes
  2024-Q4: 20 imágenes


In [8]:
# Ajustar: empezar desde 2018 y filtrar trimestres vacíos
composites = []
for year in range(2018, 2026):
    for quarter in range(1, 5):
        composites.append(quarterly_composite(year, quarter))

quarterly_col = ee.ImageCollection(composites)
print(f"Composites trimestrales: {len(composites)} (2018-2025)")

# Exportar a Google Drive
for year in range(2018, 2026):
    for q in range(1, 5):
        ms = (q - 1) * 3 + 1
        img = quarterly_composite(year, q)
        
        task = ee.batch.Export.image.toDrive(
            image=img,
            description=f'CGSM_indices_{year}_Q{q}',
            folder='CGSM_data',
            region=aoi,
            scale=10,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        task.start()

print(f"Exportaciones lanzadas: {len(composites)} tareas")
print("Monitorea el progreso en: code.earthengine.google.com > Tasks")

Composites trimestrales: 32 (2018-2025)
Exportaciones lanzadas: 32 tareas
Monitorea el progreso en: code.earthengine.google.com > Tasks


In [9]:
print(ee.data.getAssetRoots())

[{'type': 'Table', 'id': 'projects/333663781584/assets/CGSM'}]


In [10]:
# Exportar composites RGB para SamGeo (3 períodos representativos)
periodos_rgb = [
    ('2019-01-01', '2019-06-30', 'CGSM_RGB_2019_S1'),  # Período temprano
    ('2021-01-01', '2021-06-30', 'CGSM_RGB_2021_S1'),  # Período medio
    ('2024-01-01', '2024-06-30', 'CGSM_RGB_2024_S1'),  # Período reciente
]

for start, end, name in periodos_rgb:
    rgb = (s2.filterDate(start, end)
           .select(['B4', 'B3', 'B2'])
           .median()
           .clip(aoi))
    
    # Escalar a 0-255 para SamGeo
    rgb_vis = rgb.unitScale(0, 3000).multiply(255).toByte()
    
    task = ee.batch.Export.image.toDrive(
        image=rgb_vis,
        description=name,
        folder='CGSM_data',
        region=aoi,
        scale=10,
        crs='EPSG:4326',
        maxPixels=1e13
    )
    task.start()
    print(f"Exportando: {name}")

print("3 RGBs lanzados. Los períodos definitivos se ajustarán en la Fase 2.")

Exportando: CGSM_RGB_2019_S1
Exportando: CGSM_RGB_2021_S1
Exportando: CGSM_RGB_2024_S1
3 RGBs lanzados. Los períodos definitivos se ajustarán en la Fase 2.


In [12]:
# Serie Landsat 8/9 (2013-2025)
def mask_landsat_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud = qa.bitwiseAnd(1 << 3).eq(0)
    shadow = qa.bitwiseAnd(1 << 4).eq(0)
    return image.updateMask(cloud.And(shadow))

def add_landsat_indices(image):
    optical = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    ndvi = optical.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    ndwi = optical.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI')
    cmri = ndvi.subtract(ndwi).rename('CMRI')
    return image.addBands([ndvi, ndwi, cmri])

l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
      .filterBounds(aoi)
      .filterDate('2013-01-01', '2025-12-31')
      .filter(ee.Filter.lt('CLOUD_COVER', 20))
      .map(mask_landsat_clouds)
      .map(add_landsat_indices))

l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
      .filterBounds(aoi)
      .filterDate('2022-01-01', '2025-12-31')
      .filter(ee.Filter.lt('CLOUD_COVER', 20))
      .map(mask_landsat_clouds)
      .map(add_landsat_indices))

landsat = l8.merge(l9)
print(f"Imágenes Landsat 8: {l8.size().getInfo()}")
print(f"Imágenes Landsat 9: {l9.size().getInfo()}")
print(f"Total Landsat: {landsat.size().getInfo()}")

Imágenes Landsat 8: 252
Imágenes Landsat 9: 76
Total Landsat: 328


In [13]:
# Exportar composites anuales Landsat (2013-2025)
for year in range(2013, 2026):
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year, 12, 31)
    
    composite = (landsat.filterDate(start, end)
                .select(['NDVI', 'NDWI', 'CMRI'])
                .median()
                .clip(aoi))
    
    task = ee.batch.Export.image.toDrive(
        image=composite,
        description=f'CGSM_Landsat_indices_{year}',
        folder='CGSM_data',
        region=aoi,
        scale=30,
        crs='EPSG:4326',
        maxPixels=1e13
    )
    task.start()
    print(f"Exportando Landsat {year}")

print(f"\n{13} tareas Landsat lanzadas")

Exportando Landsat 2013
Exportando Landsat 2014
Exportando Landsat 2015
Exportando Landsat 2016
Exportando Landsat 2017
Exportando Landsat 2018
Exportando Landsat 2019
Exportando Landsat 2020
Exportando Landsat 2021
Exportando Landsat 2022
Exportando Landsat 2023
Exportando Landsat 2024
Exportando Landsat 2025

13 tareas Landsat lanzadas


In [14]:
# Guardar el polígono AOI detallado como GeoJSON
coords = aoi.coordinates().getInfo()[0]

geojson = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {"name": "CGSM_AOI", "source": "Delimitación detallada CGSM"},
        "geometry": {"type": "Polygon", "coordinates": [coords]}
    }]
}

with open('../data/raw/cgsm_aoi.geojson', 'w') as f:
    json.dump(geojson, f, indent=2)

print("GeoJSON actualizado con polígono detallado")
print(f"Vértices: {len(coords)}")


GeoJSON actualizado con polígono detallado
Vértices: 34


In [16]:
# --- Exportar RGBs para SamGeo ---
periodos = {
    'degradacion': {'fecha_inicio': '2020-07-01', 'fecha_fin': '2020-12-31'},
    'recuperacion': {'fecha_inicio': '2022-01-01', 'fecha_fin': '2022-06-30'},
    'actual': {'fecha_inicio': '2024-07-01', 'fecha_fin': '2025-06-30'},
}

for periodo, info in periodos.items():
    rgb = (s2.filterDate(info['fecha_inicio'], info['fecha_fin'])
           .select(['B4', 'B3', 'B2'])
           .median()
           .clip(aoi))
    
    rgb_vis = rgb.unitScale(0, 3000).multiply(255).toByte()
    
    task = ee.batch.Export.image.toDrive(
        image=rgb_vis,
        description=f'CGSM_RGB_{periodo}',
        folder='CGSM_data',
        region=aoi,
        scale=10,
        crs='EPSG:4326',
        maxPixels=1e13
    )
    task.start()
    print(f"Exportando RGB {periodo}: {info['fecha_inicio']} a {info['fecha_fin']}")

print("\n3 RGBs lanzados a Google Drive (carpeta CGSM_data)")

Exportando RGB degradacion: 2020-07-01 a 2020-12-31
Exportando RGB recuperacion: 2022-01-01 a 2022-06-30
Exportando RGB actual: 2024-07-01 a 2025-06-30

3 RGBs lanzados a Google Drive (carpeta CGSM_data)


In [18]:
tasks = ee.batch.Task.list()
for task in tasks:
    desc = task.config.get('description', '')
    if 'RGB' in desc:
        print(f"  {desc}: {task.state}")

  CGSM_RGB_actual: READY
  CGSM_RGB_recuperacion: READY
  CGSM_RGB_degradacion: READY
  CGSM_RGB_2024_S1: READY
  CGSM_RGB_2021_S1: READY
  CGSM_RGB_2019_S1: READY
